<a href="https://colab.research.google.com/github/monlkebuh/monlkebuh.github.io/blob/main/SadTalker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title **setup（about 5 minutes）**
!update-alternatives --install /usr/local/bin/python3 python3 /usr/bin/python3.8 2
!update-alternatives --install /usr/local/bin/python3 python3 /usr/bin/python3.9 1
!python --version
!apt-get update
!apt install software-properties-common
!sudo dpkg --remove --force-remove-reinstreq python3-pip python3-setuptools python3-wheel
!apt-get install python3-pip

print('Git clone project and install requirements...')
!git clone https://github.com/cedro3/SadTalker.git &> /dev/null
%cd SadTalker
!export PYTHONPATH=/content/SadTalker:$PYTHONPATH
!python3.8 -m pip install torch==1.12.1+cu113 torchvision==0.13.1+cu113 torchaudio==0.12.1 --extra-index-url https://download.pytorch.org/whl/cu113
!apt update
!apt install ffmpeg &> /dev/null
!python3.8 -m pip install -r requirements.txt


In [ ]:
#@title **download model（about 1 minute)**
print('Download pre-trained models...')
!rm -rf checkpoints
!bash scripts/download_models.sh

In [ ]:
#@title **inference for face**
image ='full3.png' #@param {type:"string"}
audio ='eluosi.wav' #@param {type:"string"}
source_image = 'examples/source_image/' + image
driven_audio = 'examples/driven_audio/' + audio

!python3.8 inference.py --driven_audio $driven_audio \
           --source_image $source_image \
           --result_dir ./results --enhancer gfpgan

In [ ]:
#@title **play movie**
import glob
from IPython.display import HTML
from base64 import b64encode
import os, sys

# get the last from results
mp4_name = sorted(glob.glob('./results/*.mp4'))[-1]

mp4 = open('{}'.format(mp4_name),'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

print('Display animation: {}'.format(mp4_name), file=sys.stderr)
display(HTML("""
  <video width=256 controls>
        <source src="%s" type="video/mp4">
  </video>
  """ % data_url))

In [ ]:
#@title **inference for portrait**
image ='full3.png' #@param {type:"string"}
audio ='eluosi.wav' #@param {type:"string"}
source_image = 'examples/source_image/' + image
driven_audio = 'examples/driven_audio/' + audio

!python3.8 inference.py --driven_audio $driven_audio \
           --source_image $source_image \
           --result_dir ./results --still --preprocess full --enhancer gfpgan

In [ ]:
#@title **play movie**
import glob
from IPython.display import HTML
from base64 import b64encode
import os, sys

# get the last from results
mp4_name = sorted(glob.glob('./results/*.mp4'))[-1]

mp4 = open('{}'.format(mp4_name),'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

print('Display animation: {}'.format(mp4_name), file=sys.stderr)
display(HTML("""
  <video width=256 controls>
        <source src="%s" type="video/mp4">
  </video>
  """ % data_url))

In [1]:
# The Super Install - Run this first!
!pip install kornia==0.6.8
!pip install facexlib
!pip install yacs
!pip install gfpgan
!pip install "numpy<2.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 551.1/551.1 kB 21.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 16.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 70.3 MB/s eta 0:00:00
  Created wheel for filterpy: filename=filterpy-1.4.5-py3-none-any.whl size=110460 sha256=bf2079eeff9251ba9a3691cae0ca71369ec7c8af9ec5d09448d9e80a588025ac
  Stored in directory: /root/.cache/pip/wheels/77/bf/4c/b0c3f4798a0166668752312a67118b27a3cd341e13ac0ae6ee
Successfully built filterpy
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4


KeyboardInterrupt: 

In [ ]:
import os
from moviepy.editor import AudioFileClip

original_audio = '/content/SadTalker/examples/driven_audio/justdance.wav'
output_folder = '/content/audio_segments/'
os.makedirs(output_folder, exist_ok=True)

audio = AudioFileClip(original_audio)
total_duration = audio.duration
first_segment_end = 13.0  # Photo 1 sings until 32 seconds in

# 1. Create the first long segment for Photo 1
seg1 = audio.subclip(0, first_segment_end)
seg1.write_audiofile(f"{output_folder}seg_001.wav", codec='pcm_s16le')

# 2. Divide the rest of the song for the other 83 photos
remaining_duration = total_duration - first_segment_end
other_seg_length = remaining_duration / 83

for i in range(1, 84): # Starts from the 2nd photo
    start = first_segment_end + ((i-1) * other_seg_length)
    end = first_segment_end + (i * other_seg_length)
    if end > total_duration: end = total_duration

    segment = audio.subclip(start, end)
    segment.write_audiofile(f"{output_folder}seg_{i+1:03d}.wav", codec='pcm_s16le')

print("Audio sliced! Photo 1 has the long intro; others have equal slices.")

In [ ]:
import os

# 1. SURGERY: Fix the internal AI code to work with modern libraries
print("--- Performing surgery on AI source code... ---")

# Fix 1: The 'functional_tensor' error
!sed -i 's/from torchvision.transforms.functional_tensor import rgb_to_grayscale/from torchvision.transforms.functional import rgb_to_grayscale/' /usr/local/lib/python3.12/dist-packages/basicsr/data/degradations.py

# Fix 2: The 'np.float' error
!sed -i 's/np.float/float/g' /usr/local/lib/python3.12/dist-packages/facexlib/alignment/awing_arch.py

# Fix 3: The 'inhomogeneous shape' error (The most stubborn one)
!sed -i 's/trans_params = np.array(\[w0, h0, s, t\[0\], t\[1\]\])/trans_params = np.array(\[w0, h0, s, t[0], t[1]\], dtype=object)/' /content/SadTalker/src/face3d/util/preprocess.py

# 2. PATHS
image_folder = '/content/SadTalker/examples/source_image/'
audio_folder = '/content/audio_segments/'
result_dir = '/content/SadTalker/results'
os.makedirs(result_dir, exist_ok=True)

images = sorted([f for f in os.listdir(image_folder) if f.endswith(('.jpg', '.png', '.jpeg'))])
audios = sorted([f for f in os.listdir(audio_folder) if f.endswith('.wav')])

# 3. START THE RELAY
%cd /content/SadTalker
for i in range(len(images)):
    img_path = os.path.join(image_folder, images[i])
    aud_path = os.path.join(audio_folder, audios[i])
    print(f"\n--- Rendering Photo {i+1} of 84: {images[i]} ---")
    !python3 inference.py --driven_audio "$aud_path" --source_image "$img_path" --result_dir "$result_dir" --still --preprocess full --enhancer gfpgan

--- Performing surgery on AI source code... ---
/content/SadTalker

--- Rendering Photo 1 of 84: 01.jpg ---
using safetensor as default
3DMM Extraction for source image
landmark Det:: 100% 1/1 [00:00<00:00,  8.20it/s]
3DMM Extraction In Video:: 100% 1/1 [00:00<00:00,  6.02it/s]
mel:: 100% 325/325 [00:00<00:00, 48417.59it/s]
audio2exp:: 100% 33/33 [00:00<00:00, 240.80it/s]
Face Renderer:: 100% 163/163 [01:55<00:00,  1.42it/s]
The generated video is named /content/SadTalker/results/2026_05_11_02.45.58/01##seg_001.mp4
OpenCV: FFMPEG: tag 0x5634504d/'MP4V' is not supported with codec id 12 and format 'mp4 / MP4 (MPEG-4 Part 14)'
OpenCV: FFMPEG: fallback to use tag 0x7634706d/'mp4v'
seamlessClone:: 100% 325/325 [00:02<00:00, 141.25it/s]
The generated video is named /content/SadTalker/results/2026_05_11_02.45.58/01##seg_001_full.mp4
face enhancer....
Face Enhancer:: 100% 325/325 [01:52<00:00,  2.89it/s]
IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resiz

In [ ]:
import os

result_dir = './relay_results'
all_subdirs = [os.path.join(result_dir, d) for d in os.listdir(result_dir) if os.path.isdir(os.path.join(result_dir, d))]
latest_subdir = max(all_subdirs, key=os.path.getmtime)

vids = sorted([f for f in os.listdir(latest_subdir) if f.endswith('.mp4')])

with open('concat_list.txt', 'w') as f:
    for v in vids:
        f.write(f"file '{os.path.join(latest_subdir, v)}'\n")

!ffmpeg -f concat -safe 0 -i concat_list.txt -c copy final_relay_with_intro.mp4